# Sliced evaluation

Issue: #15

## Question

An average hides failures. Does a model only work for popular items, or for long sessions, or for a few categories? We report the metric per slice, with an interval and a session count.

## Data used

No OTTO data yet (the sample from #4 and the ranking queries from #10 do not exist). The functions run on **synthetic** per-session results whose columns match what #10 will provide: `category`, `session_length`, `target_item`. Once the popularity baseline (#17) exists, run the same code on its per-session NDCG.

Code: `notebooks/slicing.py` (`from slicing import ...`), which uses `notebooks/bootstrap.py`. Stays out of `src/ranking/` until #52.

In [ ]:
import sys

import numpy as np
import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from slicing import check_partition, length_bucket, popularity_buckets, sliced_metrics

## Synthetic per-session results

By construction the model is better for popular items and worse for short sessions, so the slices should show it.

In [ ]:
rng = np.random.default_rng(0)
n_sessions = 20_000
n_items = 5_000

# Training-window item counts (Zipf-like). Buckets come from these only.
train_counts = {i: int(1 + 10_000 / (i + 1) ** 1.1) for i in range(n_items)}
bucket_of = popularity_buckets(train_counts)

probs = np.array([train_counts[i] for i in range(n_items)], dtype=float)
probs /= probs.sum()
target = rng.choice(n_items, size=n_sessions, p=probs)
length = np.clip(rng.geometric(0.15, size=n_sessions), 1, 60)
category = rng.integers(0, 60, size=n_sessions)

base = np.where(np.array([bucket_of[t] for t in target]) == "head", 0.45, 0.20)
base = base + np.where(length >= 4, 0.10, -0.05)
ndcg = np.clip(base + rng.normal(0, 0.15, size=n_sessions), 0, 1)

df = pl.DataFrame(
    {
        "session": np.arange(n_sessions),
        "category": [f"cat_{c:02d}" for c in category],
        "length_bucket": [length_bucket(n) for n in length],
        "popularity_bucket": [bucket_of[t] for t in target],
        "ndcg10": ndcg,
    }
)
print(df.head(3))

## Popularity buckets

Edges are computed from the training counts only: head = the items that make up the first 50 percent of events, torso = the next 30 percent, tail = the last 20 percent.

In [ ]:
share = {}
for item, b in bucket_of.items():
    share[b] = share.get(b, 0) + train_counts[item]
total = sum(share.values())
print({b: f"{v / total:.0%} of events, {sum(1 for x in bucket_of.values() if x == b)} items" for b, v in share.items()})

## Slice by session length, popularity and category

In [ ]:
by_length = sliced_metrics(df, "length_bucket", "ndcg10")
by_pop = sliced_metrics(df, "popularity_bucket", "ndcg10")
by_cat = sliced_metrics(df, "category", "ndcg10", n_boot=300)

for name, table in [("length", by_length), ("popularity", by_pop), ("category", by_cat)]:
    assert check_partition(table, df)  # counts add up to the number of sessions
    assert table.columns == ["slice", "n", "metric", "low", "high", "flag"]
    print(name, table.height, "slices")

with pl.Config(tbl_rows=12):
    print(by_length)
    print(by_pop)
    print(by_cat.sort("n").head(8))

## Small slices are flagged, not hidden

In [ ]:
small = df.filter(pl.col("category").is_in(["cat_00", "cat_01"])).with_columns(
    pl.when(pl.col("category") == "cat_00")
    .then(pl.lit("cat_00"))
    .otherwise(pl.lit("cat_01"))
    .alias("category")
)
small = pl.concat([small.filter(pl.col("category") == "cat_00"), small.filter(pl.col("category") == "cat_01").head(40)])
table = sliced_metrics(small, "category", "ndcg10", min_n=100)
print(table)
flagged = table.filter(pl.col("flag") == "low n")
assert flagged.height == 1 and flagged["slice"][0] == "cat_01"
assert table.height == 2  # still present

# missing slice values are an error
broken = df.with_columns(pl.when(pl.col("session") == 0).then(None).otherwise(pl.col("category")).alias("category"))
try:
    sliced_metrics(broken, "category", "ndcg10")
except ValueError as err:
    print("ValueError:", err)
else:
    raise AssertionError

## What the popularity slice reveals

A model can have a good overall average and still be poor on the tail. Behaviour-based labels favour popular items, because popular items are clicked more often and so appear more in the labels. The popularity slice shows how much of the average comes from head items. Here the head bucket is far ahead of the tail, so the overall number overstates how well the model works on rarely seen items. A real model must be compared on the tail slice too, not just on the mean.

## Result

Three tables (length, popularity, category) with columns slice, n, metric, low, high, flag. Session counts add up to the total in each. Slices under 100 sessions carry a `low n` flag. Edges for popularity come from training counts only.

To do when #10 and #17 exist: run the same three calls on the popularity baseline's per-session results.

## What I learned

To be written by the owner of the project.